In [8]:
from torch.utils.data import Dataset, DataLoader
from src.language_models.dictionary_corpus import Corpus
from src.language_models.utils import get_batch, batchify
import torch

In [3]:
data = '/scratch2/mrenaudin/colorlessgreenRNNs/english_data'
corpus = Corpus(data, save_tokenized=False)


In [5]:
train_data = batchify(corpus.train, 10, 'cpu')


In [6]:
for batch, i in enumerate(range(0, train_data.size(0) - 1, 35)):

        data, targets = get_batch(train_data, i, 35)
        print(data.shape)
        print(targets.shape)
        break

torch.Size([35, 10])
torch.Size([350])


In [13]:
print(targets.view(-1).shape)

torch.Size([350])


In [18]:
class BPTTDataset(Dataset):
    def __init__(self, data, bptt):
        self.data = data  # Keep on CPU initially
        self.bptt = bptt
        
    def __len__(self):
        return max(1, (len(self.data) - 1) // self.bptt)
    
    def __getitem__(self, idx):
        i = idx * self.bptt
        seq_len = min(self.bptt, len(self.data) - 1 - i)
        
        # Return consecutive tokens (classic language modeling)
        data = self.data[i:i + seq_len]
        target = self.data[i + 1:i + 1 + seq_len]
        
        return data, target

def collate_fn(batch):
    """Custom collate function to handle variable sequence lengths"""
    # batch is a list of (data, target) tuples
    data_list, target_list = zip(*batch)
    
    # Stack into batches - shape: (batch_size, seq_len)
    data_batch = torch.stack(data_list, dim=1)
    target_batch = torch.stack(target_list, dim=1)
    
    return data_batch, target_batch

# Create dataloaders with proper batching
def create_dataloaders(corpus, bptt, batch_size):
    eval_batch_size = 10
    
    # Create datasets
    train_dataset = BPTTDataset(corpus.train, bptt)
    val_dataset = BPTTDataset(corpus.valid, bptt)
    test_dataset = BPTTDataset(corpus.test, bptt)
    
    # Create dataloaders - NOW with proper batch_size
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size,  # Real batch size here
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        collate_fn=collate_fn,
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=eval_batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        collate_fn=collate_fn
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=eval_batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        collate_fn=collate_fn
    )
    
    return train_loader, val_loader, test_loader

In [19]:
train, val, test = create_dataloaders(corpus, 35, 10)

In [20]:
for batch_idx, (data, targets) in enumerate(train):
        data, targets = data.to('cpu', non_blocking=True), targets.to('cpu', non_blocking=True)
        print(data.shape)
        print(targets.shape)
        break

torch.Size([35, 10])
torch.Size([35, 10])


In [21]:
len(train)

237310